# Trial Batch Runner — Risk-Function Subtrials

Same shape as `trial_slow_batch_runner.ipynb`, but the per-tick target speed is computed by an arbitrary user-supplied function of **ground-truth risk** rather than a fixed multiplier:

```
v_target_kmh = SPEED_MULTIPLIER * risk_speed_fn(zone_speed_kmh, gt_risk)
```

`risk_speed_fn(v, r)` is your function. `v` is the live zone speed limit (km/h) read from CARLA each tick. `r` is the **ground-truth** scene risk computed on the fly by the simulator (no model involved). This isolates the risk-aware speed-control concept from any model quality concern — if the idea works at all, it will work here.

Pick `selected_risk` in the config cell from `risk1 … risk5` (each producing the matching `riskfn1 … riskfn5` subtrial — the five slots the `trial_comparison` notebook plots across the bottom two rows of its 4×3 grid) and run the notebook once per function. Saves only the dataset.jsonl + summary.json, no images.

## 0 — CARLA Launch

In [1]:
import subprocess

AUTO_LAUNCH_CARLA = True
CARLA_EXE = "CarlaUE4.exe"

if AUTO_LAUNCH_CARLA:
    subprocess.Popen(CARLA_EXE, shell=True)
    print(f"Launched {CARLA_EXE}")
else:
    print("AUTO_LAUNCH_CARLA is False. Assuming CARLA is already running.")

Launched CarlaUE4.exe


## 1 — Imports

In [2]:
import sys
import time
from pathlib import Path

from MIREIA.config import Config
from MIREIA.core.speed_policy import KineticRiskSpeedPolicy
from MIREIA.simulation.trials import EgoTrialConfig, TrialDefinition, TrialRunner

## 2 — Discover Trials

In [4]:
trials_root = Path(Config.PATH_TO_TRIALS)
all_trials = sorted(p.parent.name for p in trials_root.glob("*/trial.json") if p.is_file())

# Filtering (leave INCLUDE_PREFIXES empty to include every trial)
INCLUDE_PREFIXES: list[str] = []   # e.g. ["auto_"] to only run auto-generated trials
EXCLUDE_TRIALS:   list[str] = []   # exact trial names to skip

if INCLUDE_PREFIXES:
    selected_trial_names = [n for n in all_trials if any(n.startswith(p) for p in INCLUDE_PREFIXES)]
else:
    selected_trial_names = list(all_trials)
selected_trial_names = [n for n in selected_trial_names[:] if n not in EXCLUDE_TRIALS]

print(f"Found {len(all_trials)} trial(s); {len(selected_trial_names)} selected to run:")
for n in selected_trial_names:
    print(f"  - {n}")

Found 20 trial(s); 20 selected to run:
  - auto_17A_WetNoon_Town05_HighVol
  - auto_17B_WetNoon_Town05_LowVol
  - auto_17C_WetNoon_Town10HD_HighVol
  - auto_17D_WetNoon_Town10HD_LowVol
  - auto_18A_MidRainyNoon_Town05_HighVol
  - auto_18B_MidRainyNoon_Town05_LowVol
  - auto_18C_MidRainyNoon_Town10HD_HighVol
  - auto_18D_MidRainyNoon_Town10HD_LowVol
  - auto_19A_CloudySunset_Town05_HighVol
  - auto_19B_CloudySunset_Town05_LowVol
  - auto_19C_CloudySunset_Town10HD_HighVol
  - auto_19D_CloudySunset_Town10HD_LowVol
  - auto_20A_SoftRainSunset_Town05_HighVol
  - auto_20B_SoftRainSunset_Town05_LowVol
  - auto_20C_SoftRainSunset_Town10HD_HighVol
  - auto_20D_SoftRainSunset_Town10HD_LowVol
  - auto_21A_ClearNoon_Town05_HighVol_NoFog_Night
  - auto_21B_CloudyNoon_Town05_LowVol_NoFog_Night
  - auto_21C_WetNoon_Town10HD_HighVol_NoFog_Night
  - auto_21D_HardRainNoon_Town10HD_LowVol_NoFog_Night


## 3 — Risk Function & Subtrial Config

`risk_speed_fn(v, r)` returns the desired target speed (km/h) for that tick.

- `v` (`v_target`) — live zone speed limit (km/h), read from CARLA every tick.
- `r` — ground-truth scene risk; can be `None` if the oracle fails momentarily.

### Formula

$$v_{safe} = \sqrt{\, v_{min}^2 + \bigl(v_{target}^2 - v_{min}^2\bigr) \cdot e^{-\lambda \cdot \max(0,\; R - R_{base})} \,}$$

| Parameter | Variable | Role |
|-----------|----------|------|
| `R_BASE` | $R_{base}$ | Risk below this → exponent = 0 → $v_{safe} = v_{target}$ (no penalty). Raise it to ignore moderate-risk events |
| `LAMBDA` | $\lambda$ | Aggressiveness: higher = steeper drop above the baseline |
| `V_MIN_FACTOR` | $v_{min} = $ `V_MIN_FACTOR` $ \times v_{target}$ | Speed floor as a fraction of the zone limit; guarantees the car never fully stops |

**Energy scaling:** the decay is applied to squared velocities so the first unit of excess risk removes the maximum possible kinetic energy — steeper non-linear response than a linear clamp.

Set `selected_risk` below to one of `risk1 … risk5` (subtrial names `riskfn1 … riskfn5`). The preview after the cell samples the function at realistic GT-risk values (≈ 1–10+) so you can tune `R_BASE`, `LAMBDA`, and `V_MIN_FACTOR` before launching the batch.

In [5]:

risk1 = {
    "SUBTRIAL_NAME": "riskfn1",
    "SPEED_MULTIPLIER": 1.0,
    "MAX_STEPS_PER_TRIAL": 6000,
    "R_BASE": 3.0,           # ignore risk below this baseline (no slowdown)
    "LAMBDA": 1,             # response aggressiveness (higher = steeper drop above R_BASE)
    "V_MIN_FACTOR": 0.75,    # v_min = V_MIN_FACTOR * v_target (speed floor)
}

risk2 = {  # The least aggressive setting
    "SUBTRIAL_NAME": "riskfn2",
    "SPEED_MULTIPLIER": 1.0,
    "MAX_STEPS_PER_TRIAL": 6000,
    "R_BASE": 3.0,
    "LAMBDA": 0.5,
    "V_MIN_FACTOR": 0.85,
}

risk3 = {
    "SUBTRIAL_NAME": "riskfn3",
    "SPEED_MULTIPLIER": 1.0,
    "MAX_STEPS_PER_TRIAL": 6000,
    "R_BASE": 3.0,
    "LAMBDA": 2,
    "V_MIN_FACTOR": 0.55,
}

risk4 = {  # Delayed onset, sharp response: ignore moderate risk, react hard above R=5
    "SUBTRIAL_NAME": "riskfn4",
    "SPEED_MULTIPLIER": 1.0,
    "MAX_STEPS_PER_TRIAL": 6000,
    "R_BASE": 5.0,
    "LAMBDA": 2.5,
    "V_MIN_FACTOR": 0.55,
}

risk5 = {  # Same delayed onset as fn4 but cuts deeper when engaged (floor 0.45 instead of 0.55).
           # Tests whether the high-risk shortfall vs fn3 in heavy-rain scenarios (21D) closes
           # when we go below fn3's floor in the high-R regime.
    "SUBTRIAL_NAME": "riskfn5",
    "SPEED_MULTIPLIER": 1.0,
    "MAX_STEPS_PER_TRIAL": 6000,
    "R_BASE": 5.0,
    "LAMBDA": 2.5,
    "V_MIN_FACTOR": 0.45,
}



selected_risk = risk5
SUBTRIAL_NAME = selected_risk["SUBTRIAL_NAME"]
SPEED_MULTIPLIER = selected_risk["SPEED_MULTIPLIER"]
MAX_STEPS_PER_TRIAL = selected_risk["MAX_STEPS_PER_TRIAL"]
R_BASE = selected_risk["R_BASE"]
LAMBDA = selected_risk["LAMBDA"]
V_MIN_FACTOR = selected_risk["V_MIN_FACTOR"]

risk_speed_fn = KineticRiskSpeedPolicy(r_base=R_BASE, lam=LAMBDA, v_min_factor=V_MIN_FACTOR)

# Sanity-check the shape at realistic GT-risk values before launching.
print(risk_speed_fn.preview())

ego_cfg = EgoTrialConfig(
    name=SUBTRIAL_NAME,
    ego_blueprint="vehicle.lincoln.mkz_2020",
    # target_speed_kmh=None -> controller uses the live zone speed limit as v_target.
    speed_multiplier=SPEED_MULTIPLIER,
    use_vehicle_camera_defaults=True,
    controller_mode="behavior_agent",
    controller_behavior="normal",
)

runner = TrialRunner(verbose=False)
print(f"Risk-fn ego config ready: subtrial='{SUBTRIAL_NAME}', SPEED_MULTIPLIER={SPEED_MULTIPLIER}")

KineticRiskSpeedPolicy  r_base=5.0  lam=2.5  v_min_factor=0.45
   v_target      R ->   v_safe   ratio
       30.0   None ->    30.00   1.000
       30.0    0.0 ->    30.00   1.000
       30.0    4.0 ->    30.00   1.000
       30.0    5.0 ->    30.00   1.000
       30.0    6.0 ->    15.53   0.518
       30.0    8.0 ->    13.51   0.450
       30.0   12.0 ->    13.50   0.450
       30.0   22.0 ->    13.50   0.450

       50.0   None ->    50.00   1.000
       50.0    0.0 ->    50.00   1.000
       50.0    4.0 ->    50.00   1.000
       50.0    5.0 ->    50.00   1.000
       50.0    6.0 ->    25.88   0.518
       50.0    8.0 ->    22.52   0.450
       50.0   12.0 ->    22.50   0.450
       50.0   22.0 ->    22.50   0.450

       90.0   None ->    90.00   1.000
       90.0    0.0 ->    90.00   1.000
       90.0    4.0 ->    90.00   1.000
       90.0    5.0 ->    90.00   1.000
       90.0    6.0 ->    46.59   0.518
       90.0    8.0 ->    40.54   0.450
       90.0   12.0 ->    40.50   0.450

## 4 — Run Each Trial

In [6]:
all_summaries = []
failed: list[tuple[str, str]] = []

# --- Run-time accounting (sim vs wall, cap meaning) ----------------------------
# step  = one CARLA world.tick() = one fixed_delta seconds of simulated time.
# image_stride controls JSONL cadence, NOT loop iteration count.
# The loop exits early when controller.done() — ego reached the route end (within
# arrival_threshold_m = 3 m). max_steps is the cap; functions that slow the car
# more cover the same distance in MORE sim time, so they need a higher cap.
fixed_delta_s    = float(Config.SIM_FIXED_DELTA_SECONDS)
stride_ticks     = int(Config.RECORD_EVERY_N_TICKS)
sim_time_cap_s   = MAX_STEPS_PER_TRIAL * fixed_delta_s
max_jsonl_frames = MAX_STEPS_PER_TRIAL // stride_ticks
record_hz        = 1.0 / (fixed_delta_s * stride_ticks)

print("Run configuration:")
print(f"  subtrial_name:        '{SUBTRIAL_NAME}'")
print(f"  SPEED_MULTIPLIER:     {SPEED_MULTIPLIER:.3f}")
print(f"  risk source:          ground truth (wm.get_risk() per tick)")
print(f"  fixed_delta:          {fixed_delta_s:.3f} s/tick  ({1/fixed_delta_s:.1f} Hz physics)")
print(f"  image_stride:         {stride_ticks} ticks/frame  ({record_hz:.2f} Hz JSONL records)")
print(f"  max_steps_per_trial:  {MAX_STEPS_PER_TRIAL}  (cap = {sim_time_cap_s:.1f} s sim time, "
      f"<= {max_jsonl_frames} JSONL frames)")
print(f"  finished=True   -> controller reported done before the cap (good)")
print(f"  finished=False  -> hit max_steps cap (raise MAX_STEPS_PER_TRIAL or make risk_speed_fn faster)")

_progress_state: dict = {"trial_started": 0.0}

def _format_secs(s: float) -> str:
    if s < 60:
        return f"{s:5.1f}s"
    m, sec = divmod(s, 60)
    return f"{int(m):02d}:{int(sec):02d}"

def _on_progress(step: int, max_steps: int) -> None:
    now = time.time()
    last = _progress_state.get("last_print", 0.0)
    if now - last < 0.2:
        return
    _progress_state["last_print"] = now
    elapsed = now - _progress_state["trial_started"]
    rate = step / elapsed if elapsed > 0 else 0.0
    pct = 100.0 * step / max_steps if max_steps else 0.0
    eta_to_cap = (max_steps - step) / rate if rate > 0 else float("inf")
    speedup = (step * fixed_delta_s) / elapsed if elapsed > 0 else 0.0
    sys.stdout.write(
        f"\r    step {step:5d}/{max_steps} ({pct:5.1f}% of cap)  "
        f"wall={_format_secs(elapsed)}  "
        f"rate={rate:5.1f} t/s  "
        f"speedup={speedup:4.2f}x  "
        f"eta_to_cap={_format_secs(eta_to_cap) if rate > 0 else '  inf'}    "
    )
    sys.stdout.flush()

batch_started = time.time()
for idx, trial_name in enumerate(selected_trial_names, start=1):
    print()
    print(f">>> [{idx}/{len(selected_trial_names)}] {trial_name}", flush=True)
    _progress_state["trial_started"] = time.time()
    _progress_state["last_print"] = 0.0
    try:
        trial = TrialDefinition.load(trial_name)
        summary = runner.run_subtrial(
            trial=trial,
            ego_cfg=ego_cfg,
            max_steps=MAX_STEPS_PER_TRIAL,
            image_stride=Config.RECORD_EVERY_N_TICKS,
            store_rgb_images=False,
            store_topdown_images=False,
            store_risk_frame_images=False,
            store_static_risk_map=False,
            draw_debug_every_tick=False,
            risk_speed_fn=risk_speed_fn,
            use_ground_truth_risk=True,
            progress_callback=_on_progress,
            progress_every_n_steps=25,
        )
        elapsed = time.time() - _progress_state["trial_started"]
        sim_seconds = summary.num_frames * fixed_delta_s * stride_ticks
        speedup = sim_seconds / elapsed if elapsed > 0 else float("inf")
        sys.stdout.write("\r" + " " * 130 + "\r")
        sys.stdout.flush()
        print(
            f"    ok in {elapsed:6.1f}s wall  "
            f"frames={summary.num_frames:4d}  "
            f"sim_t={sim_seconds:6.1f}s  "
            f"speedup={speedup:5.2f}x  "
            f"dist={summary.traveled_m:7.1f}m  "
            f"risk_auc={summary.risk_auc:.3f}  "
            f"finished={summary.finished}"
        )
        all_summaries.append(summary)
    except Exception as e:
        elapsed = time.time() - _progress_state["trial_started"]
        sys.stdout.write("\r" + " " * 130 + "\r")
        sys.stdout.flush()
        print(f"    FAILED in {elapsed:.1f}s: {type(e).__name__}: {e}")
        failed.append((trial_name, f"{type(e).__name__}: {e}"))

batch_elapsed = time.time() - batch_started
print()
print(f"Batch done in {batch_elapsed:.1f}s (ok={len(all_summaries)}, failed={len(failed)})")

Run configuration:
  subtrial_name:        'riskfn5'
  SPEED_MULTIPLIER:     1.000
  risk source:          ground truth (wm.get_risk() per tick)
  fixed_delta:          0.050 s/tick  (20.0 Hz physics)
  image_stride:         5 ticks/frame  (4.00 Hz JSONL records)
  max_steps_per_trial:  6000  (cap = 300.0 s sim time, <= 1200 JSONL frames)
  finished=True   -> controller reported done before the cap (good)
  finished=False  -> hit max_steps cap (raise MAX_STEPS_PER_TRIAL or make risk_speed_fn faster)

>>> [1/20] auto_17A_WetNoon_Town05_HighVol


: 

## 5 — Summary

In [ ]:
if not all_summaries and not failed:
    print("No trials were run.")
else:
    print(f"=== Successful runs ({len(all_summaries)}) ===")
    print(
        f"  {'trial_name':50s} {'frames':>6s} {'dist_m':>9s} "
        f"{'risk_auc':>9s} {'risk/m':>9s}  finished  run_path"
    )
    for s in all_summaries:
        print(
            f"  {s.trial_name:50s} {s.num_frames:6d} {s.traveled_m:9.1f} "
            f"{s.risk_auc:9.3f} {s.risk_per_meter:9.5f}  {str(s.finished):>8s}  {s.run_path}"
        )

    if failed:
        print()
        print(f"=== Failed runs ({len(failed)}) ===")
        for name, err in failed:
            print(f"  {name}: {err}")

=== Successful runs (1) ===
  trial_name                                         frames    dist_m  risk_auc    risk/m  finished  run_path
  auto_17A_WetNoon_Town05_HighVol                       394     549.1   244.631   2.55819      True  t:\TFG\MIREIA\trials\auto_17A_WetNoon_Town05_HighVol\runs\20260513_235509_riskfn4


: 